# SB100 Squad 2 — extração com o pipeline híbrido

**Docling lê as páginas com camada de texto e sem tabela** (21% do acervo);
o Chandra lê o resto. O PyMuPDF entra como rede de segurança nas páginas do
Docling, recuperando parágrafo que ele derruba.

Mais rápido que só-Chandra, e nas páginas de texto não tem como alucinar,
porque lê o texto já embutido no PDF em vez de adivinhar pela imagem.

Antes de começar: **Ambiente de execução → Alterar tipo → GPU**.

Se a sessão cair, nada do que já subiu se perde. Reabra e rode de novo:
a seleção é por `extracted = false`, então ele traz só o que falta.


## 1. Preparo

Vai pedir o arquivo `.env`, que está em `SB100/squad-2\`. Nenhuma chave
é digitada; o arquivo fica só nesta sessão.


In [ ]:
import os, shutil, torch
from google.colab import files

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - troque em Ambiente de execucao")

%cd /content
!rm -rf /content/repo /content/squad-2
# o repositório da organização tem outros projetos: baixa só a pasta do site (~1 MB)
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/CCDSB100/squad-2.git /content/squad-2
!git -C /content/squad-2 sparse-checkout set 2026/parser_rag_framework
!mv /content/squad-2/2026/parser_rag_framework /content/repo && rm -rf /content/squad-2
%cd /content/repo
!pip install -q supabase python-dotenv pymupdf

print()
print("Escolha o arquivo .env da pasta SB100/squad-2:")
enviados = files.upload()
nome = next(iter(enviados))
if nome != ".env": shutil.move(f"/content/repo/{nome}", "/content/repo/.env")

linhas = [l for l in open("/content/repo/.env") if "=" in l and not l.startswith("#")]
tem = {l.split("=")[0].strip() for l in linhas}
for k in ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"):
    assert k in tem, f"o .env enviado nao tem {k}"
print(f"ambiente pronto: {len(linhas)} variaveis")


## 2. Ver o que falta

Só relata, não baixa nada.


In [ ]:
!python extractor/baixar_lote.py --quantos 0


## 3. Baixar o lote

Prioriza o que foi pedido pelo botão **Analisar** do site; depois os
menores, que dão retorno rápido e cabem numa sessão sem risco de perder
tudo no meio. Imprime a estimativa de GPU antes de você seguir.


In [ ]:
QUANTOS = 3    #@param {type:"integer"}
MENORES = True #@param {type:"boolean"}

extra = "--menores" if MENORES else ""
!python extractor/baixar_lote.py --quantos {QUANTOS} {extra}


## 4. Extrair

**CHANDRA_MAX_LADO** vale para as páginas que forem pelo Chandra.

**LOTE_PAGINAS** não tem efeito aqui: a batelada só existe no caminho
só-Chandra.


In [ ]:
# extrator: hibrido
CHANDRA_MAX_LADO = 2200  #@param {type:"integer"}
LOTE_PAGINAS = 1         #@param {type:"integer"}

import os
os.environ["CHANDRA_MAX_LADO"] = str(CHANDRA_MAX_LADO)
os.environ["LOTE_PAGINAS"] = str(LOTE_PAGINAS)
!python extractor/extrator_hibrido.py


## 5. Mandar para o Supabase

**Dois jeitos, escolha um:**

- **Rodar esta célula.** O Colab grava direto no banco.
- **Pular esta célula** e rodar a 7, que baixa `extracao_chandra.zip`. Depois
  envie o zip na aba **Extração** do site, em *Enviar extração pronta*. O site
  confere o pacote antes de gravar e mostra o que é novo e o que já existe.

Nos dois, documento **já extraído é pulado**: regravar apaga páginas e blocos
que podem ter sido curados à mão. Para regravar de propósito, use
`--substituir` aqui, ou marque a opção no site.

A lógica é a mesma nos dois caminhos (`api/extracao.py`): atualiza o
documento no lugar, não encosta em `article_metadata`, e gera os recortes
de figura na sequência.


In [ ]:
!python scripts/ingerir_extracao.py /content/export


## 6. Conferir

Depois disto, abra o site: os documentos do lote aparecem como extraídos,
com as figuras.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12


## 7. Guardar uma cópia (opcional)

O resultado já está no Supabase. Isto é só para ter o arquivo bruto da
corrida na sua máquina.


In [ ]:
import shutil
shutil.make_archive('/content/extracao_hibrido', 'zip', '/content/export')
from google.colab import files
files.download('/content/extracao_hibrido.zip')
